In [0]:
# =====================================================
# PARAMÉTRAGE - Widgets pour exécution via Databricks Job
# =====================================================

dbutils.widgets.text("catalog_name", "banking_lakehouse", "Catalog Unity Catalog")
dbutils.widgets.text("environment", "dev", "Environnement (dev/staging/prod)")

CATALOG = dbutils.widgets.get("catalog_name")
ENVIRONMENT = dbutils.widgets.get("environment")

print(f"✅ Paramètres reçus : catalog={CATALOG}, environment={ENVIRONMENT}")

In [0]:
# =====================================================
# Notebook : 05_build_dim_date
# Objectif : Construire la dimension Date partagée
#            entre les 2 Data Domains (Client / Transactions)
# Pattern  : Dimension standard Kimball
# =====================================================

from pyspark.sql.functions import (
    col, explode, sequence, to_date, date_format,
    year, quarter, month, dayofmonth, dayofweek, dayofyear,
    weekofyear, when, lit
)

TABLE_DIM_DATE = f"{CATALOG}.gold.dim_date"

# --- Génération d'une séquence de dates ---
# On couvre une plage large : 2015 à 2027 (couvre nos données historiques + marge future)
df_date_range = spark.sql("""
    SELECT explode(sequence(to_date('2015-01-01'), to_date('2027-12-31'), interval 1 day)) AS full_date
""")

df_dim_date = (
    df_date_range
    .withColumn("date_sk", date_format(col("full_date"), "yyyyMMdd").cast("int"))  # Surrogate key lisible
    .withColumn("year", year(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("day_of_month", dayofmonth(col("full_date")))
    .withColumn("day_of_week", dayofweek(col("full_date")))  # 1=dimanche, 7=samedi
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
    .withColumn("week_of_year", weekofyear(col("full_date")))
    .withColumn(
        "is_weekend",
        when(col("day_of_week").isin([1, 7]), lit(True)).otherwise(lit(False))
    )
)

print(f"📊 Nombre de dates générées : {df_dim_date.count()}")
display(df_dim_date.limit(10))

In [0]:
# =====================================================
# Écriture Gold - dim_date
# =====================================================

df_dim_date.write.format("delta").mode("overwrite").saveAsTable(TABLE_DIM_DATE)

print(f"✅ Table {TABLE_DIM_DATE} créée avec succès")

# Vérification rapide
spark.table(TABLE_DIM_DATE).select(
    "date_sk", "full_date", "year", "quarter", "month_name", "day_name", "is_weekend"
).show(5)